# Assignment 5 Resilient Colab Scaffold

This notebook is a setup and validation companion for the Assignment 5 scaffold.
It clones or refreshes the repo, mounts Google Drive, initializes W&B, prepares
local and Drive-backed output paths, and includes smoke-check cells you can run
in Colab after you finish the implementation.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/stanford-cs336/assignment5-alignment.git"
REPO_DIR = Path("/content/assignment5-alignment")
BRANCH = "main"

WANDB_PROJECT = "cs336-assignment5-alignment"
WANDB_ENTITY = None
RUN_NAME = None

DRIVE_ROOT = Path("/content/drive/MyDrive/cs336-assignment5")
RUN_SUBDIR = "resilient-run"

SAVE_EVERY_STEPS = 10
EVAL_EVERY_STEPS = 10
RESUME_FROM_CHECKPOINT = None

LOCAL_OUTPUT_DIR = REPO_DIR / "outputs"
LOCAL_CHECKPOINT_DIR = LOCAL_OUTPUT_DIR / "checkpoints"
LOCAL_EVAL_DIR = LOCAL_OUTPUT_DIR / "eval"
LOCAL_WANDB_DIR = LOCAL_OUTPUT_DIR / "wandb"

DRIVE_RUN_DIR = DRIVE_ROOT / RUN_SUBDIR
DRIVE_CHECKPOINT_DIR = DRIVE_RUN_DIR / "checkpoints"
DRIVE_EVAL_DIR = DRIVE_RUN_DIR / "eval"
DRIVE_WANDB_DIR = DRIVE_RUN_DIR / "wandb"


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import os
import subprocess

def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if REPO_DIR.exists():
    print(f'Refreshing existing repo at {REPO_DIR}')
    run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'])
    run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH])
    run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH])
else:
    print(f'Cloning repo into {REPO_DIR}')
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])
    run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH])

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())


In [ ]:
import sys
import subprocess

run_base = [sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip']
subprocess.run(run_base, check=True)

# Try the optional GPU wheel first, then continue with the no-deps fallback if it
# is unavailable in the current Colab runtime.
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'flash-attn'], check=True)
    print('flash-attn install attempted successfully')
except Exception:
    print('flash-attn unavailable; continuing with the no-deps fallback')

deps = [
    'wandb',
    'accelerate',
    'transformers',
    'torch',
    'tqdm',
    'typer',
    'xopen',
    'math-verify',
    'pylatexenc',
    'jupyter',
    'notebook',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', *deps], check=True)

# Install the repo without forcing flash-attn so Colab can proceed even if the
# optional GPU-specific wheel is unavailable in the current runtime.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)


In [ ]:
for path in [
    LOCAL_OUTPUT_DIR,
    LOCAL_CHECKPOINT_DIR,
    LOCAL_EVAL_DIR,
    LOCAL_WANDB_DIR,
    DRIVE_RUN_DIR,
    DRIVE_CHECKPOINT_DIR,
    DRIVE_EVAL_DIR,
    DRIVE_WANDB_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)
    print('created', path)


In [ ]:
import wandb

wandb.login()
wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=RUN_NAME,
    dir=str(LOCAL_WANDB_DIR),
    tags=['assignment5', 'colab', 'resilient'],
    resume='allow',
)

print('W&B run:', wandb_run.name if wandb_run else None)


## Persistence And Resume

Future training code should save checkpoints every `SAVE_EVERY_STEPS`, write W&B
metrics continuously, and mirror new checkpoints into Google Drive after each
save event. If Colab disconnects, you can resume from an explicit checkpoint or
from the newest checkpoint discovered in Drive.


In [ ]:
from pathlib import Path
import shutil

def latest_checkpoint(checkpoint_root: Path) -> Path | None:
    if not checkpoint_root.exists():
        return None
    candidates = [p for p in checkpoint_root.iterdir() if p.is_dir()]
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)

def resolve_resume_checkpoint(
    explicit_resume_path: str | Path | None = None,
    drive_checkpoint_root: Path = DRIVE_CHECKPOINT_DIR,
    local_checkpoint_root: Path = LOCAL_CHECKPOINT_DIR,
) -> Path | None:
    if explicit_resume_path:
        return Path(explicit_resume_path)
    drive_latest = latest_checkpoint(drive_checkpoint_root)
    if drive_latest is not None:
        return drive_latest
    return latest_checkpoint(local_checkpoint_root)

def mirror_checkpoint_to_drive(local_checkpoint_path: Path) -> Path:
    destination = DRIVE_CHECKPOINT_DIR / local_checkpoint_path.name
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(local_checkpoint_path, destination)
    return destination

print('latest drive checkpoint:', latest_checkpoint(DRIVE_CHECKPOINT_DIR))
print('resume candidate:', resolve_resume_checkpoint(RESUME_FROM_CHECKPOINT))


In [ ]:
from cs336_alignment.config import (
    CheckpointConfig,
    DriveSyncConfig,
    EvalConfig,
    ExpertIterationConfig,
    GRPOConfig,
    SFTConfig,
    WandbConfig,
)

wandb_config = WandbConfig(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    run_name=RUN_NAME,
    tags=['assignment5', 'colab', 'resilient'],
    log_dir=LOCAL_WANDB_DIR,
)
checkpoint_config = CheckpointConfig(
    output_dir=LOCAL_CHECKPOINT_DIR,
    save_every_steps=SAVE_EVERY_STEPS,
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
)
drive_sync_config = DriveSyncConfig(
    enabled=True,
    drive_root=DRIVE_ROOT,
    per_run_subdir=RUN_SUBDIR,
)
eval_config = EvalConfig(
    output_dir=LOCAL_EVAL_DIR,
    batch_size=1,
    temperature=1.0,
    top_p=1.0,
    max_tokens=1024,
    stop_tokens=['</answer>'],
)
sft_config = SFTConfig(
    train_steps=0,
    gradient_accumulation_steps=1,
    eval_every_steps=EVAL_EVERY_STEPS,
    save_every_steps=SAVE_EVERY_STEPS,
    wandb=wandb_config,
    checkpoint=checkpoint_config,
    drive_sync=drive_sync_config,
)
grpo_config = GRPOConfig(
    train_steps=0,
    rollout_batch_size=0,
    train_batch_size=0,
    gradient_accumulation_steps=1,
    eval_every_steps=EVAL_EVERY_STEPS,
    save_every_steps=SAVE_EVERY_STEPS,
    wandb=wandb_config,
    checkpoint=checkpoint_config,
    drive_sync=drive_sync_config,
)
expert_iteration_config = ExpertIterationConfig(
    n_ei_steps=0,
    rollout_batch_size=0,
    rollouts_per_example=0,
    sft_epochs_per_round=0,
    eval_every_steps=EVAL_EVERY_STEPS,
    save_every_steps=SAVE_EVERY_STEPS,
    wandb=wandb_config,
    checkpoint=checkpoint_config,
    drive_sync=drive_sync_config,
)

print(wandb_config)
print(checkpoint_config)
print(drive_sync_config)


In [ ]:
import cs336_alignment
import cs336_alignment.config as config_module
import cs336_alignment.sft as sft_module
import cs336_alignment.grpo as grpo_module
import cs336_alignment.evaluation as evaluation_module
import tests.adapters as adapters_module

print('cs336_alignment:', cs336_alignment)
print('config:', config_module)
print('sft:', sft_module)
print('grpo:', grpo_module)
print('evaluation:', evaluation_module)
print('adapters:', adapters_module)


In [ ]:
from cs336_alignment import (
    run_zero_shot_baseline,
    run_sft_training,
    run_expert_iteration,
    run_grpo_training,
)

print(run_zero_shot_baseline)
print(run_sft_training)
print(run_expert_iteration)
print(run_grpo_training)


In [ ]:
import shutil

from cs336_alignment.config import CheckpointConfig, DriveSyncConfig, SFTConfig, WandbConfig
from transformers import AutoTokenizer

DEMO_MODEL_DIR = LOCAL_OUTPUT_DIR / 'tiny-gpt2-colab'
if not DEMO_MODEL_DIR.exists():
    DEMO_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    source_model_dir = REPO_DIR / 'tests/fixtures/tiny-gpt2'
    for source_path in source_model_dir.iterdir():
        if source_path.is_file():
            shutil.copy2(source_path, DEMO_MODEL_DIR / source_path.name)
    AutoTokenizer.from_pretrained('gpt2').save_pretrained(DEMO_MODEL_DIR)

sft_demo_config = SFTConfig(
    train_steps=1,
    learning_rate=1e-4,
    train_batch_size=1,
    gradient_accumulation_steps=1,
    normalize_constant=1.0,
    eval_every_steps=1,
    save_every_steps=1,
    wandb=WandbConfig(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        run_name='sft-demo',
        tags=['assignment5', 'colab', 'resilient', 'demo', 'sft'],
        log_dir=LOCAL_WANDB_DIR,
    ),
    checkpoint=CheckpointConfig(
        output_dir=LOCAL_CHECKPOINT_DIR / 'sft-demo',
        save_every_steps=1,
        max_checkpoints=1,
        resume_from_checkpoint=None,
    ),
    drive_sync=DriveSyncConfig(
        enabled=True,
        drive_root=DRIVE_ROOT,
        per_run_subdir=f'{RUN_SUBDIR}-sft-demo',
    ),
)

sft_demo_result = run_sft_training(
    model_id=str(DEMO_MODEL_DIR),
    dataset_path=REPO_DIR / 'tests/fixtures/sft_sample.jsonl',
    validation_dataset_path=None,
    config=sft_demo_config,
)
print(sft_demo_result)


In [ ]:
import json
import shutil

from cs336_alignment.config import CheckpointConfig, DriveSyncConfig, EvalConfig, GRPOConfig, WandbConfig
from transformers import AutoTokenizer

if not DEMO_MODEL_DIR.exists():
    DEMO_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    source_model_dir = REPO_DIR / 'tests/fixtures/tiny-gpt2'
    for source_path in source_model_dir.iterdir():
        if source_path.is_file():
            shutil.copy2(source_path, DEMO_MODEL_DIR / source_path.name)
    AutoTokenizer.from_pretrained('gpt2').save_pretrained(DEMO_MODEL_DIR)

toy_grpo_path = LOCAL_OUTPUT_DIR / 'toy_grpo_train.jsonl'
toy_grpo_examples = [
    {'prompt': 'State the capital of France.', 'answer': 'Paris'},
    {'prompt': 'What color is the sky on a clear day?', 'answer': 'blue'},
]
with open(toy_grpo_path, 'w', encoding='utf-8') as f:
    for example in toy_grpo_examples:
        f.write(json.dumps(example) + '\n')

def toy_reward_fn(response, ground_truth):
    reward = 1.0 if str(ground_truth).strip().lower() in str(response).strip().lower() else 0.0
    return {'reward': reward, 'format_reward': reward, 'answer_reward': reward}

grpo_demo_config = GRPOConfig(
    train_steps=1,
    learning_rate=1e-4,
    rollout_batch_size=2,
    group_size=1,
    train_batch_size=2,
    gradient_accumulation_steps=1,
    epochs_per_rollout_batch=1,
    loss_type='reinforce_with_baseline',
    eval_every_steps=1,
    save_every_steps=1,
    wandb=WandbConfig(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        run_name='grpo-demo',
        tags=['assignment5', 'colab', 'resilient', 'demo', 'grpo'],
        log_dir=LOCAL_WANDB_DIR,
    ),
    checkpoint=CheckpointConfig(
        output_dir=LOCAL_CHECKPOINT_DIR / 'grpo-demo',
        save_every_steps=1,
        max_checkpoints=1,
        resume_from_checkpoint=None,
    ),
    drive_sync=DriveSyncConfig(
        enabled=True,
        drive_root=DRIVE_ROOT,
        per_run_subdir=f'{RUN_SUBDIR}-grpo-demo',
    ),
)

grpo_demo_result = run_grpo_training(
    model_id=str(DEMO_MODEL_DIR),
    train_dataset_path=toy_grpo_path,
    validation_dataset_path=toy_grpo_path,
    reward_fn=toy_reward_fn,
    config=grpo_demo_config,
    eval_config=EvalConfig(output_dir=LOCAL_EVAL_DIR / 'grpo-demo', batch_size=1, temperature=0.8, top_p=1.0, max_tokens=16),
)
print(grpo_demo_result)


## What To Run In Colab

Run the SFT demo cell first, then the GRPO demo cell. Both build a local tiny
model directory from the repo fixture weights plus a usable tokenizer, so you
can verify training, checkpointing, and Drive sync before swapping in your own
model or dataset paths.
